In [0]:
# Databricks notebook: 07_top_users
# Purpose: Transform PhonePe top user JSON into a ranked
# state/district/pincode Silver Delta table.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    BooleanType,
    LongType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
SILVER_TABLE = "workspace.silver.top_users"

# 34 country files + 1,224 state files
EXPECTED_SOURCE_FILES = 1_258

print("Configuration")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")


# ============================================================
# 2. SELECT TOP USER SOURCE FILES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE).alias("b")

top_user_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "top") &
        (F.col("domain") == "user")
    )
    .alias("m")
)

top_user_source_df = (
    top_user_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

actual_source_files = (
    top_user_source_df
    .select("source_path")
    .distinct()
    .count()
)

print("\nSource validation")
print(f"Expected source files: {EXPECTED_SOURCE_FILES:,}")
print(f"Actual source files  : {actual_source_files:,}")

assert actual_source_files == EXPECTED_SOURCE_FILES, (
    f"Expected {EXPECTED_SOURCE_FILES:,} files, "
    f"but found {actual_source_files:,}."
)

print("Top user source validation passed.")


# ============================================================
# 3. DEFINE SCHEMA
# Supports historical and current field names.
# ============================================================

top_user_record_schema = StructType([
    StructField("name", StringType(), True),
    StructField("entityName", StringType(), True),
    StructField("registeredUsers", LongType(), True),
    StructField("registeredCount", LongType(), True)
])

top_user_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),

    StructField(
        "data",
        StructType([
            StructField(
                "states",
                ArrayType(top_user_record_schema),
                True
            ),
            StructField(
                "districts",
                ArrayType(top_user_record_schema),
                True
            ),
            StructField(
                "pincodes",
                ArrayType(top_user_record_schema),
                True
            )
        ]),
        True
    ),

    StructField("responseTimestamp", LongType(), True)
])

print("\nExplicit top user schema created.")


# ============================================================
# 4. PARSE JSON
# ============================================================

parsed_top_user_df = (
    top_user_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            top_user_schema
        )
    )
)

parse_summary = (
    parsed_top_user_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col("parsed_json.data.states").isNull() &
                F.col("parsed_json.data.districts").isNull() &
                F.col("parsed_json.data.pincodes").isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_top_data")
    )
    .first()
)

print("\nJSON parsing")
print(f"Source files          : {parse_summary['source_files']:,}")
print(
    f"Files without top data: "
    f"{parse_summary['files_without_top_data']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_top_data"] == 0, (
    "One or more files do not contain readable state, "
    "district, or pincode user ranking data."
)

print("Top user JSON parsing passed.")


# ============================================================
# 5. FUNCTION TO EXTRACT EACH RANKING LEVEL
# ============================================================

def extract_user_ranking(
    source_df,
    array_name,
    geography_level
):
    return (
        source_df

        # Preserve original ranking order.
        .select(
            "*",
            F.posexplode(
                F.col(f"parsed_json.data.{array_name}")
            ).alias("rank_index", "rank_record")
        )

        .select(
            F.col("file_scope"),

            F.lit(geography_level)
                .alias("geography_level"),

            F.when(
                F.col("file_scope") == "state",
                F.col("state_slug")
            )
            .otherwise(F.lit(None).cast("string"))
            .alias("parent_state_slug"),

            F.coalesce(
                F.trim(F.col("rank_record.name")),
                F.trim(F.col("rank_record.entityName"))
            ).alias("entity_name"),

            F.lower(
                F.regexp_replace(
                    F.coalesce(
                        F.trim(F.col("rank_record.name")),
                        F.trim(F.col("rank_record.entityName"))
                    ),
                    "[^a-zA-Z0-9]+",
                    "-"
                )
            ).alias("entity_slug"),

            (F.col("rank_index") + F.lit(1))
                .cast("int")
                .alias("rank_position"),

            F.col("year").cast("int").alias("year"),
            F.col("quarter").cast("int").alias("quarter"),
            F.col("period"),

            F.coalesce(
                F.col("rank_record.registeredUsers"),
                F.col("rank_record.registeredCount")
            )
            .cast("long")
            .alias("registered_user_count"),

            F.when(
                F.col("rank_record.registeredUsers").isNotNull(),
                F.lit("registeredUsers")
            )
            .when(
                F.col("rank_record.registeredCount").isNotNull(),
                F.lit("registeredCount")
            )
            .otherwise(F.lit(None))
            .alias("registered_count_source_field"),

            F.col("parsed_json.responseTimestamp")
                .alias("response_timestamp_epoch_ms"),

            F.col("relative_path"),
            F.col("source_path"),
            F.col("source_commit"),
            F.col("content_sha256"),
            F.col("ingested_at_utc")
        )
    )


# ============================================================
# 6. EXTRACT STATE, DISTRICT, AND PINCODE RANKINGS
# ============================================================

top_user_states_df = extract_user_ranking(
    parsed_top_user_df,
    "states",
    "state"
)

top_user_districts_df = extract_user_ranking(
    parsed_top_user_df,
    "districts",
    "district"
)

top_user_pincodes_df = extract_user_ranking(
    parsed_top_user_df,
    "pincodes",
    "pincode"
)

top_users_final_df = (
    top_user_states_df
    .unionByName(top_user_districts_df)
    .unionByName(top_user_pincodes_df)
)

print("\nState, district, and pincode user rankings extracted.")


# ============================================================
# 7. DATA-QUALITY VALIDATION
# ============================================================

quality_result = (
    top_users_final_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("geography_level").isNull() |
                F.col("entity_name").isNull() |
                F.col("rank_position").isNull() |
                F.col("registered_user_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_required_values"),

        F.sum(
            F.when(
                F.col("registered_user_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_registered_user_counts"),

        F.sum(
            F.when(
                F.col("rank_position") <= 0,
                1
            ).otherwise(0)
        ).alias("invalid_rank_positions"),

        F.countDistinct("source_path")
            .alias("represented_source_files"),

        F.min("registered_user_count")
            .alias("minimum_registered_users"),

        F.max("registered_user_count")
            .alias("maximum_registered_users")
    )
    .first()
)

duplicate_rank_keys = (
    top_users_final_df
    .groupBy(
        "source_path",
        "geography_level",
        "rank_position"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nData-quality results")
print(f"Total Silver rows              : {quality_result['total_rows']:,}")
print(
    f"Missing required values        : "
    f"{quality_result['missing_required_values']:,}"
)
print(
    f"Negative registered-user counts: "
    f"{quality_result['negative_registered_user_counts']:,}"
)
print(
    f"Invalid ranking positions      : "
    f"{quality_result['invalid_rank_positions']:,}"
)
print(
    f"Source files represented       : "
    f"{quality_result['represented_source_files']:,}"
)
print(
    f"Minimum registered users       : "
    f"{quality_result['minimum_registered_users']:,}"
)
print(
    f"Maximum registered users       : "
    f"{quality_result['maximum_registered_users']:,}"
)
print(f"Duplicate ranking keys         : {duplicate_rank_keys:,}")

assert quality_result["total_rows"] > 0
assert quality_result["missing_required_values"] == 0
assert quality_result["negative_registered_user_counts"] == 0
assert quality_result["invalid_rank_positions"] == 0
assert quality_result["represented_source_files"] == EXPECTED_SOURCE_FILES
assert duplicate_rank_keys == 0

print("Top user quality checks passed.")


# ============================================================
# 8. DISPLAY SOURCE-FIELD AND GEOGRAPHY COVERAGE
# ============================================================

print("\nRows by geography level and source field")

display(
    top_users_final_df
    .groupBy(
        "geography_level",
        "registered_count_source_field"
    )
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count")
    )
    .orderBy(
        "geography_level",
        "registered_count_source_field"
    )
)


# ============================================================
# 9. SAVE MANAGED DELTA TABLE
# ============================================================

(
    top_users_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\nSilver Delta table created: {SILVER_TABLE}")


# ============================================================
# 10. VALIDATE SAVED TABLE
# ============================================================

saved_top_users_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_top_users_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period")
    )
    .first()
)

print("\nSaved-table validation")
print(f"Saved rows              : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period            : {saved_summary['first_period']}")
print(f"Last period             : {saved_summary['last_period']}")

assert saved_summary["row_count"] > 0
assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final top user Delta-table validation passed.")


# ============================================================
# 11. DISPLAY SAMPLE
# ============================================================

display(
    saved_top_users_df
    .select(
        "geography_level",
        "parent_state_slug",
        "entity_name",
        "rank_position",
        "year",
        "quarter",
        "period",
        "registered_user_count",
        "registered_count_source_field"
    )
    .orderBy(
        "year",
        "quarter",
        "geography_level",
        "parent_state_slug",
        "rank_position"
    )
    .limit(30)
)

Configuration
Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.top_users
Expected files: 1,258

Source validation
Expected source files: 1,258
Actual source files  : 1,258
Top user source validation passed.

Explicit top user schema created.

JSON parsing
Source files          : 1,258
Files without top data: 0
Top user JSON parsing passed.

State, district, and pincode user rankings extracted.

Data-quality results
Total Silver rows              : 10,880
Missing required values        : 0
Negative registered-user counts: 0
Invalid ranking positions      : 0
Source files represented       : 1,258
Minimum registered users       : 78
Maximum registered users       : 90,563,992
Duplicate ranking keys         : 0
Top user quality checks passed.

Rows by geography level and source field


geography_level,registered_count_source_field,row_count,source_file_count
district,registeredCount,10540,1258
state,registeredCount,340,34



Silver Delta table created: workspace.silver.top_users

Saved-table validation
Saved rows              : 10,880
Source files represented: 1,258
First period            : 2018-Q1
Last period             : 2026-Q2
Final top user Delta-table validation passed.


geography_level,parent_state_slug,entity_name,rank_position,year,quarter,period,registered_user_count,registered_count_source_field
district,null,bengaluru urban district,1,2018,1,2018-Q1,2169190,registeredCount
district,null,pune district,2,2018,1,2018-Q1,1345147,registeredCount
district,null,jaipur district,3,2018,1,2018-Q1,848946,registeredCount
district,null,rangareddy district,4,2018,1,2018-Q1,818216,registeredCount
district,null,medchal malkajgiri district,5,2018,1,2018-Q1,717914,registeredCount
district,null,mumbai suburban district,6,2018,1,2018-Q1,716230,registeredCount
district,null,thane district,7,2018,1,2018-Q1,650285,registeredCount
district,null,ahmedabad district,8,2018,1,2018-Q1,597372,registeredCount
district,null,gurugram district,9,2018,1,2018-Q1,542519,registeredCount
district,null,gautambuddha nagar district,10,2018,1,2018-Q1,530855,registeredCount


# Silver Top Users — Completion Summary

The `workspace.silver.top_users` Delta table was successfully created and validated.

## Results

- Total ranked records: **10,880**
- State-ranking records: **340**
- District-ranking records: **10,540**
- Source files represented: **1,258**
- Data coverage: **2018-Q1 to 2026-Q2**

## Table grain

Each record represents:

- One source geography
- One ranking geography level
- One ranking position
- One year and quarter

## Available data

- State and district user rankings
- Registered-user count
- Ranking position
- Original source-field name
- Complete source lineage

## Source-schema handling

The pipeline supports both `registeredUsers` and `registeredCount`. The current pinned source version uses `registeredCount`.

The current source does not provide pincode-level user rankings.

## Analytical use

This table supports:

- Regional adoption leaderboards
- Rank-movement analysis
- Persistent high-adoption region detection
- Emerging digital-market identification
- Regional ranking-stability research